<a href="https://colab.research.google.com/github/Lingadarini/trendpulse/blob/part2-numpy/part2_numpy_pandas/part2_numpy_pandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone -b part2-numpy https://github.com/Lingadarini/trendpulse.git
!ls trendpulse/part2_numpy_pandas/data

Cloning into 'trendpulse'...
remote: Enumerating objects: 56, done.
remote: Counting objects: 100% (56/56), done.
remote: Compressing objects: 100% (46/46), done.
remote: Total 56 (delta 19), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (56/56), 2.05 MiB | 3.84 MiB/s, done.
Resolving deltas: 100% (19/19), done.
categories.csv	GBvideos.csv  INvideos.csv  USvideos.csv


# Part 2 - NumPy and Pandas

In [ ]:
import numpy as np
import pandas as pd

DATA_DIR = "trendpulse/part2_numpy_pandas/data/"
COUNTRIES = ["US", "GB", "IN"]

frames = {}
for country in COUNTRIES:
    df_c = pd.read_csv(f"{DATA_DIR}{country}videos.csv")
    df_c["country"] = country
    frames[country] = df_c
    print(f"{country}: shape = {df_c.shape}")

combined = pd.concat(frames.values(), ignore_index=True)
print("Combined shape:", combined.shape)
print()
print(combined[["views", "likes", "dislikes", "comment_count"]].dtypes)

US: shape = (16080, 17)
GB: shape = (15075, 17)
IN: shape = (15075, 17)
Combined shape: (46230, 17)

views              int64
likes            float64
dislikes           int64
comment_count    float64
dtype: object


## NumPy arrays

In [ ]:
# Fill blanks with 0 first, so the arrays have no NaN values
views = combined["views"].to_numpy()
likes = combined["likes"].fillna(0).to_numpy()
dislikes = combined["dislikes"].to_numpy()
comments = combined["comment_count"].fillna(0).to_numpy()

for name, arr in [("views", views), ("likes", likes),
                  ("dislikes", dislikes), ("comments", comments)]:
    print(f"{name:9s} shape={arr.shape}  dtype={arr.dtype}")

# Indexing and slicing
top10 = np.sort(views)[::-1][:10]
print("\n10 largest view counts:", top10)

print("Every 100th record (first 5):", views[::100][:5])
print("Number of every-100th records:", len(views[::100]))

sub = views[1000:1010]
print("Sub-array views[1000:1010]:", sub)

views     shape=(46230,)  dtype=int64
likes     shape=(46230,)  dtype=float64
dislikes  shape=(46230,)  dtype=int64
comments  shape=(46230,)  dtype=float64

10 largest view counts: [360760164 343297002 286714357 282812154 255362869 236275528 230704137
 210747294 200444300 181785943]
Every 100th record (first 5): [ 48555  21869 245258 421352  54521]
Number of every-100th records: 463
Sub-array views[1000:1010]: [ 414515   88881  155347  631406 1261551  798063  930369  341965  277706
 2943874]


## Engagement score (NumPy only)

In [ ]:
# (a) Safe ratios: np.where avoids dividing by zero
votes = likes + dislikes
like_ratio = np.where(votes > 0, likes / np.where(votes > 0, votes, 1), 0.0)
comment_rate = np.where(views > 0, comments / np.where(views > 0, views, 1), 0.0)

# (b) Min-max normalisation: scales each array to the range 0..1
def min_max(arr):
    span = arr.max() - arr.min()
    return (arr - arr.min()) / span if span > 0 else np.zeros_like(arr, dtype=float)

views_norm = min_max(views.astype(float))
likes_norm = min_max(likes)
comments_norm = min_max(comments)

print("like_ratio   : min", like_ratio.min().round(3), " max", like_ratio.max().round(3))
print("comment_rate : min", comment_rate.min().round(5), " max", comment_rate.max().round(5))
print("views_norm   : min", views_norm.min(), " max", views_norm.max())
print("likes_norm   : min", likes_norm.min(), " max", likes_norm.max())
print("comments_norm: min", comments_norm.min(), " max", comments_norm.max())

like_ratio   : min 0.0  max 1.0
comment_rate : min 0.0  max 0.02463
views_norm   : min 0.0  max 1.0
likes_norm   : min 0.0  max 1.0
comments_norm: min 0.0  max 1.0


In [ ]:
# (c) Weighted engagement score
W_VIEWS, W_LIKES, W_COMMENTS = 0.2, 0.4, 0.4
engagement = (W_VIEWS * views_norm
              + W_LIKES * likes_norm
              + W_COMMENTS * comments_norm)

# (d) Log-scaled views (log1p handles zeros safely)
log_views = np.log1p(views)

# Boolean mask: top 10% by engagement
cutoff = np.percentile(engagement, 90)
top10_mask = engagement >= cutoff

print("Weights:", W_VIEWS, W_LIKES, W_COMMENTS, "(sum =", W_VIEWS + W_LIKES + W_COMMENTS, ")")
print("Engagement: min", engagement.min().round(4), " max", engagement.max().round(4))
print("90th percentile cutoff:", cutoff.round(5))
print("Records flagged top 10%:", top10_mask.sum(), "of", len(engagement))
print("Raw views range     :", f"{views.min():,} to {views.max():,}")
print("Log views range     :", log_views.min().round(2), "to", log_views.max().round(2))

Weights: 0.2 0.4 0.4 (sum = 1.0 )
Engagement: min 0.0  max 0.9076
90th percentile cutoff: 0.00567
Records flagged top 10%: 4623 of 46230
Raw views range     : 1,066 to 360,760,164
Log views range     : 6.97 to 19.7


### Why these weights

| Component | Weight | Reason |
|---|---|---|
| Views (normalised) | 0.2 | Views show reach, but a video can be watched by many people who do not care about it, so views get the smallest weight |
| Likes (normalised) | 0.4 | A like is an active choice by a viewer, so it shows approval |
| Comments (normalised) | 0.4 | A comment takes the most effort, so it is the strongest sign of real engagement |

The weights add up to 1, so the score stays between 0 and 1.

### Why raw views are hard to plot
Views are extremely skewed: most videos have a few hundred thousand views, but a handful have hundreds of millions. On a normal axis these few huge values stretch the scale and squash almost every other video into one corner of the chart. Taking the logarithm (log1p) compresses the range so the full spread of videos becomes visible.

## Reshape and broadcasting

In [ ]:
# Convert the trending_date strings (yy.dd.mm) to real dates
combined["trending_dt"] = pd.to_datetime(combined["trending_date"], format="%y.%d.%m")

# (days x countries) table of daily trending counts, as a NumPy array
counts_table = pd.crosstab(combined["trending_dt"], combined["country"])[COUNTRIES]
counts = counts_table.to_numpy()
print("counts shape (days x countries):", counts.shape)
print("First 3 days:\n", counts[:3])

# reshape and flatten
flat = counts.flatten()
print("\nflatten shape:", flat.shape)
back = flat.reshape(counts.shape)
print("reshape back equals original:", np.array_equal(back, counts))

# Broadcasting 1: each country's share of each day's total
day_totals = counts.sum(axis=1, keepdims=True)           # shape (days, 1)
share = counts / np.where(day_totals == 0, 1, day_totals)
print("\nShares of day 1:", share[0].round(3), "(sum =", share[0].sum().round(3), ")")

# Broadcasting 2: each day's value relative to that country's average
country_avg = counts.mean(axis=0)                        # shape (countries,)
relative = counts / country_avg
print("Country averages:", country_avg.round(1))
print("Day 1 relative to average:", relative[0].round(2))

counts shape (days x countries): (80, 3)
First 3 days:
 [[200 200 200]
 [202 200 201]
 [200 200 200]]

flatten shape: (240,)
reshape back equals original: True

Shares of day 1: [0.333 0.333 0.333] (sum = 1.0 )
Country averages: [201.  188.4 188.4]
Day 1 relative to average: [1.   1.06 1.06]
